[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/pydantic-certified/notebooks/day-09-performance-patterns.ipynb#scrollTo=bb000001)

---
# Day 9 · Performance, Patterns, and Pydantic Internals
**certified-journeys / pydantic-certified** · Review · Advanced Usage

> **Goal for today:** Understand the performance levers in Pydantic v2 — frozen models, `TypeAdapter`, `model_construct()`, and `model_validate_json()` — and know when each is the right tool.


In [ ]:
%pip install -q 'pydantic>=2.0' pydantic-settings fastapi httpx


## Step 1 · `ConfigDict` — Model Configuration

**`model_config`** is the Pydantic v2 way to configure a model's runtime behaviour.
It replaces the `class Config` inner class from v1.

Key options:

| Setting | Effect |
|---|---|
| `frozen=True` | Makes instances immutable and hashable |
| `str_strip_whitespace=True` | Strips leading/trailing whitespace from strings |
| `validate_default=True` | Validates default values at class definition time |
| `extra='forbid'` | Raises on unexpected fields (strict API contracts) |
| `populate_by_name=True` | Allows both alias and real field name |

**Docs:** https://docs.pydantic.dev/latest/concepts/config/


In [ ]:
from pydantic import BaseModel, ConfigDict, Field


class PointConfig(BaseModel):
    model_config = ConfigDict(
        frozen=True,                  # immutable + hashable
        str_strip_whitespace=True,    # auto-strip string fields
        extra="forbid",               # reject unexpected keys
    )

    x: float
    y: float
    label: str = ""


p1 = PointConfig(x=1.0, y=2.0, label="  origin  ")  # whitespace stripped
p2 = PointConfig(x=3.0, y=4.0)

print(f"p1.label repr: {repr(p1.label)}")  # whitespace stripped

# Frozen models are hashable — can be used as dict keys or in sets
points = {p1, p2}
point_map = {p1: "start", p2: "end"}
print(f"Points set: {len(points)} items")
print(f"p1 in map: {point_map[p1]}")

# Frozen models reject mutation
try:
    p1.x = 99.0
except Exception as e:
    print(f"\nMutation blocked: {type(e).__name__}: {e}")

# extra='forbid' rejects unknown fields
from pydantic import ValidationError
try:
    PointConfig(x=1.0, y=2.0, z=3.0)
except ValidationError as e:
    print(f"Extra field rejected: {e.errors()[0]['msg']}")


### What just happened?

- **`frozen=True`** triggers `__hash__` generation and replaces `__setattr__` with a raise — the model is now truly immutable.
- **`str_strip_whitespace=True`** applies to all `str` fields automatically — no field-level validator needed.
- **`extra='forbid'`** turns unknown field names into validation errors — great for strict API contracts and config files.
- Frozen models can be used in `set`, `frozenset`, and as `dict` keys — useful for deduplication and caching.


## Step 2 · `model_rebuild()` — Refreshing Forward References

`model_rebuild()` forces Pydantic to re-resolve all forward references and re-generate
the core schema. You need it when:

1. A model has a **self-referential field** (recursive types).
2. A model references **another model that was defined later**.
3. You **monkey-patched or dynamically modified** a model after definition.

**Docs:** https://docs.pydantic.dev/latest/concepts/postponed_annotations/#self-referential-models


In [ ]:
from __future__ import annotations
from typing import Optional, List
from pydantic import BaseModel


class Category(BaseModel):
    name: str
    parent: Optional[Category] = None        # forward self-reference
    subcategories: List[Category] = []       # list of self-references


# Without model_rebuild(), this would raise PydanticUserError on complex nesting
Category.model_rebuild()

data = {
    "name": "Electronics",
    "subcategories": [
        {
            "name": "Computers",
            "subcategories": [
                {"name": "Laptops"},
                {"name": "Desktops"},
            ],
        },
        {"name": "Phones"},
    ],
}

electronics = Category.model_validate(data)
computers = electronics.subcategories[0]
print(f"Root: {electronics.name}")
print(f"Sub: {[s.name for s in electronics.subcategories]}")
print(f"Sub-sub: {[s.name for s in computers.subcategories]}")

# model_rebuild return value — tells you if anything changed
result = Category.model_rebuild()
print(f"\nmodel_rebuild() on already-built model: {result}")


### What just happened?

- **`model_rebuild()` returns `True`** when it actually rebuilt the schema, and `None` when the model was already complete — useful for conditional rebuilding.
- The `from __future__ import annotations` import makes all annotations strings at parse time, allowing self-references to exist before the class is fully defined.
- Calling `model_rebuild()` more than once is safe and cheap when the schema hasn't changed.
- In production, call `model_rebuild()` once at module load time — not inside hot paths.


## Step 3 · `TypeAdapter` — Validate Any Type

`TypeAdapter` lets you validate **any Python type** — not just `BaseModel` subclasses.
Use it for primitive types, dataclasses, TypedDicts, bare `list[Model]`, and Union types.

It also exposes `validate_json()` and `dump_json()` for direct JSON handling.

**Docs:** https://docs.pydantic.dev/latest/concepts/type_adapter/


In [ ]:
from typing import Annotated, List, Union
from pydantic import BaseModel, Field, TypeAdapter, ValidationError


# ── Validate a list of models without a wrapper BaseModel ────────────────────
class Product(BaseModel):
    sku: str
    price: float = Field(gt=0)


adapter = TypeAdapter(List[Product])

products_json = '[{"sku":"A1","price":9.99},{"sku":"A2","price":14.50}]'
products = adapter.validate_json(products_json)  # parse JSON + validate in one step
print(f"Parsed {len(products)} products")
print(f"First: {products[0]}")

# ── Validate primitive and annotated types ───────────────────────────────────
EmailType = Annotated[str, Field(pattern=r"^[^@]+@[^@]+\.[^@]+$")]
email_adapter = TypeAdapter(EmailType)

good_email = email_adapter.validate_python("user@example.com")
print(f"\nValid email: {good_email}")

try:
    email_adapter.validate_python("not-an-email")
except ValidationError as e:
    print(f"Invalid email: {e.errors()[0]['msg']}")

# ── Validate a Union type ─────────────────────────────────────────────────────
NumericAdapter = TypeAdapter(Union[int, float, str])
print(f"\n'42' coerced to int: {NumericAdapter.validate_python('42')}")
print(f"Type: {type(NumericAdapter.validate_python('42')).__name__}")

# ── JSON Schema from TypeAdapter ─────────────────────────────────────────────
import json
schema = adapter.json_schema()
print(f"\nList[Product] JSON schema (type): {schema.get('type') or schema.get('$defs')}")


### What just happened?

- **`TypeAdapter(List[Product])`** validates a top-level list without a wrapper `BaseModel` — cleaner API for list endpoints and streaming data.
- **`validate_json()`** on a `TypeAdapter` parses JSON and validates in one Rust-side operation — faster than `json.loads()` then `validate_python()`.
- `TypeAdapter` works on **any valid Python type annotation**: primitives, `Annotated`, `Union`, `dataclass`, `TypedDict`.
- `TypeAdapter` instances are **reusable** — create once at module level and reuse for maximum performance.


## Step 4 · `model_validate_json()` vs `json.loads()` + `model_validate()` Benchmark

Pydantic v2's `model_validate_json()` uses a Rust JSON parser internally.
It avoids creating a Python dict intermediary, which saves both CPU and memory.

**Docs:** https://docs.pydantic.dev/latest/concepts/performance/

The benchmark uses `timeit` over 10,000 iterations on a realistic payload.


In [ ]:
import json
import timeit
from typing import Optional, List
from pydantic import BaseModel, Field


class Address(BaseModel):
    street: str
    city: str
    zip_code: str


class UserBenchmark(BaseModel):
    user_id: int
    username: str
    email: str
    age: int = Field(ge=0, le=150)
    scores: List[float]
    address: Address
    bio: Optional[str] = None


PAYLOAD_JSON = json.dumps({
    "user_id": 42,
    "username": "benchmark_user",
    "email": "bench@example.com",
    "age": 30,
    "scores": [95.5, 87.2, 92.0, 88.8, 91.4],
    "address": {"street": "123 Test St", "city": "Benchtown", "zip_code": "00000"},
    "bio": "A benchmark test user with nested data and a list of scores.",
})

N = 10_000

# Method 1: model_validate_json() — single Rust-side operation
t1 = timeit.timeit(
    stmt="UserBenchmark.model_validate_json(PAYLOAD_JSON)",
    globals={"UserBenchmark": UserBenchmark, "PAYLOAD_JSON": PAYLOAD_JSON},
    number=N,
)

# Method 2: json.loads() then model_validate() — two-step, Python dict intermediary
t2 = timeit.timeit(
    stmt="UserBenchmark.model_validate(json.loads(PAYLOAD_JSON))",
    globals={"UserBenchmark": UserBenchmark, "json": json, "PAYLOAD_JSON": PAYLOAD_JSON},
    number=N,
)

print(f"Results over {N:,} iterations:")
print(f"  model_validate_json():              {t1:.3f}s  ({t1/N*1000:.3f} ms/call)")
print(f"  json.loads() + model_validate():    {t2:.3f}s  ({t2/N*1000:.3f} ms/call)")
speedup = t2 / t1
print(f"\n  model_validate_json() is ~{speedup:.1f}x faster")


### What just happened?

- **`model_validate_json()`** is consistently faster — it skips Python dict allocation by parsing JSON directly into the model in Rust.
- The speedup is typically **1.5–3x** depending on payload size and nesting depth.
- For high-throughput APIs (thousands of requests/second), this difference is significant at scale.
- The two-step approach (`json.loads` → `model_validate`) creates an unnecessary Python dict that then gets garbage-collected — wasted allocation and GC pressure.


## Step 5 · `model_construct()` — Bypassing Validation

`model_construct()` builds a model instance **without running any validation**.
Fields are set directly; no type coercion, no validators, no defaults for missing fields.

**Use only for trusted, already-validated data** — e.g., data read from your own database
that was validated on write. Never use on user-supplied input.

**Docs:** https://docs.pydantic.dev/latest/concepts/models/#model-methods-and-properties


In [ ]:
import timeit
from pydantic import BaseModel, Field


class DBRecord(BaseModel):
    record_id: int
    name: str
    value: float = Field(ge=0)
    is_valid: bool = True


# Normal construction — validates everything
normal = DBRecord(record_id=1, name="Row 1", value=42.5)
print(f"Normal:    {normal}")

# model_construct() — no validation, no coercion
fast = DBRecord.model_construct(record_id=2, name="Row 2", value=99.0, is_valid=True)
print(f"Construct: {fast}")

# DANGER: model_construct() won't catch invalid data
danger = DBRecord.model_construct(record_id=3, name="Danger", value=-999.0)  # negative — no error!
print(f"\nInvalid (no error!): value={danger.value}")
print("^ model_construct() SKIPPED the ge=0 constraint")

# Benchmark: model_validate vs model_construct
data = {"record_id": 1, "name": "bench", "value": 1.0}
N = 50_000

t_validate = timeit.timeit(
    stmt="DBRecord.model_validate(data)",
    globals={"DBRecord": DBRecord, "data": data},
    number=N,
)
t_construct = timeit.timeit(
    stmt="DBRecord.model_construct(**data)",
    globals={"DBRecord": DBRecord, "data": data},
    number=N,
)

print(f"\nBenchmark over {N:,} iterations:")
print(f"  model_validate():   {t_validate:.3f}s")
print(f"  model_construct():  {t_construct:.3f}s")
print(f"  Speedup: ~{t_validate/t_construct:.1f}x")


### What just happened?

- **`model_construct()` is much faster** than `model_validate()` — it skips the entire validation pipeline.
- The **danger** is visible: a negative `value` was accepted silently — the `ge=0` constraint was never checked.
- Only use `model_construct()` on data from **your own trusted storage** (e.g., rows returned from your own database after they passed validation on write).
- **Never** use `model_construct()` on user-supplied input, deserialized API responses from third parties, or any data path you don't fully control.


## Step 6 · Frozen Models as Cache Keys

Because `frozen=True` makes models hashable, you can use them as
function arguments in `functools.lru_cache` or as keys in `dict`/`set`.
This is a clean pattern for memoized computation over structured config.


In [ ]:
import functools
from pydantic import BaseModel, ConfigDict


class QueryConfig(BaseModel):
    model_config = ConfigDict(frozen=True)

    table: str
    limit: int = 100
    offset: int = 0
    order_by: str = "id"


@functools.lru_cache(maxsize=128)
def fetch_results(config: QueryConfig) -> list[dict]:
    """Simulates an expensive DB call; result cached by config."""
    # In production, this would be: conn.execute(f"SELECT ... FROM {config.table} ...")
    print(f"  [DB HIT] table={config.table} limit={config.limit} offset={config.offset}")
    return [{"id": i, "table": config.table} for i in range(config.offset, config.offset + config.limit)]


cfg_a = QueryConfig(table="users", limit=10)
cfg_b = QueryConfig(table="users", limit=10)  # same values → same hash
cfg_c = QueryConfig(table="orders", limit=5)

print(f"cfg_a == cfg_b: {cfg_a == cfg_b}")
print(f"hash(cfg_a) == hash(cfg_b): {hash(cfg_a) == hash(cfg_b)}")
print()

print("First call (cfg_a):")
r1 = fetch_results(cfg_a)

print("Second call (cfg_b — same values, cache hit):")
r2 = fetch_results(cfg_b)  # no "[DB HIT]" printed — served from cache

print("Third call (cfg_c — different values, cache miss):")
r3 = fetch_results(cfg_c)

print(f"\nCache info: {fetch_results.cache_info()}")


### What just happened?

- **`frozen=True` → hashable**: `cfg_a` and `cfg_b` have equal hash and equality, so `lru_cache` treats them as the same key.
- The second call with `cfg_b` (same values as `cfg_a`) is served from cache — no "DB HIT" printed.
- This pattern is powerful for memoizing expensive operations parameterized by structured config objects.
- `cache_info()` shows hits, misses, and current cache size — use for tuning `maxsize`.


In [ ]:
# Challenge: Implement a type-safe event log processor
#
# Requirements:
#   1. Define an EventRecord model:
#        event_id: int, event_type: str, payload: dict[str, float | int | str],
#        timestamp: float
#
#   2. Create a TypeAdapter for List[EventRecord]
#
#   3. Write a function process_events_json(json_str: str) -> dict that:
#        - Uses TypeAdapter.validate_json() (not json.loads + validate_python)
#        - Returns a summary: {"total": N, "by_type": {event_type: count}}
#
#   4. Create a frozen EventProcessorConfig model with:
#        batch_size: int = 100, strict_types: bool = True
#      Use it as an lru_cache key on a simulated config lookup function
#
#   5. Write a benchmark comparing validate_json() vs json.loads() + validate_python()
#      on 5000 iterations with a list of 3 events per call
#
# Sample data:
import json
sample_events_json = json.dumps([
    {"event_id": 1, "event_type": "purchase", "payload": {"amount": 49.99, "item": "widget"}, "timestamp": 1700000000.0},
    {"event_id": 2, "event_type": "view",     "payload": {"page": "home"},                   "timestamp": 1700000001.0},
    {"event_id": 3, "event_type": "purchase", "payload": {"amount": 9.99,  "item": "gadget"}, "timestamp": 1700000002.0},
])

# Your solution here
# from pydantic import TypeAdapter, BaseModel, ConfigDict
# ...


---
## Day 9 key concepts recap

| Concept | What to remember |
|---|---|
| `ConfigDict(frozen=True)` | Immutable + hashable; can be used in sets, as dict keys, with lru_cache |
| `ConfigDict(extra='forbid')` | Rejects unknown fields — strict API/config contracts |
| `model_rebuild()` | Resolves forward references; safe to call multiple times |
| `TypeAdapter(T)` | Validate any type, not just `BaseModel` — create once, reuse |
| `model_validate_json()` | ~1.5–3x faster than `json.loads()` + `model_validate()` — avoids Python dict |
| `model_construct()` | Bypasses ALL validation — trusted data only, never user input |
| `model_fields_set` | Set of field names explicitly provided at construction time |

> **Tip:** `model_construct()` bypasses all validation — it's for trusted internal data paths only. Never use it on user-supplied data.

---
## What's next
**Day 10** → Capstone — build a complete type-safe config and schema layer for an ML pipeline, combining BaseSettings, discriminated unions, computed fields, and FastAPI — all in one notebook.

Mark Day 9 complete in your [tracker](../index.html).
